In [3]:
import pandas as pd
import sqlalchemy
import pandas as pd
from sqlalchemy import create_engine
from pathlib import Path


In [4]:
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "customer_revenue_platform"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Connected Successfully")

Connected Successfully


In [5]:
customers = pd.read_sql("SELECT * FROM customers", engine)
orders = pd.read_sql("SELECT * FROM orders", engine)
payments = pd.read_sql("SELECT * FROM payments", engine)
reviews = pd.read_sql("SELECT * FROM reviews", engine)
order_items = pd.read_sql("SELECT * FROM order_items", engine)
products = pd.read_sql("SELECT * FROM products", engine)

In [6]:
datasets = {
    "customers": customers,
    "orders": orders,
    "payments": payments,
    "order_items": order_items,
    "products": products,
    "reviews": reviews
}

for name, df in datasets.items():
    print(name, df.shape)

customers (99441, 5)
orders (99441, 8)
payments (103886, 5)
order_items (112650, 7)
products (32951, 9)
reviews (99224, 7)


In [7]:
# creating master dataset by (Joining)merging all the tables on their respective keys
customer_df = (
    customers
    .merge(orders, on="customer_id", how="left")
    .merge(payments, on="order_id", how="left")
    .merge(reviews, on="order_id", how="left")
    .merge(order_items, on="order_id", how="left")
)

customer_df.shape

(119143, 28)

In [8]:
# checking the columns of the master dataset
customer_df.columns.tolist()

['customer_id',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'order_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'payment_sequential',
 'payment_type',
 'payment_installments',
 'payment_value',
 'review_id',
 'review_score',
 'review_comment_title',
 'review_comment_message',
 'review_creation_date',
 'review_answer_timestamp',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value']

In [9]:
# checking the data types and null values in the master dataset
customer_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119143 entries, 0 to 119142
Data columns (total 28 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   customer_id                    119143 non-null  str    
 1   customer_unique_id             119143 non-null  str    
 2   customer_zip_code_prefix       119143 non-null  int64  
 3   customer_city                  119143 non-null  str    
 4   customer_state                 119143 non-null  str    
 5   order_id                       119143 non-null  str    
 6   order_status                   119143 non-null  str    
 7   order_purchase_timestamp       119143 non-null  str    
 8   order_approved_at              118966 non-null  str    
 9   order_delivered_carrier_date   117057 non-null  str    
 10  order_delivered_customer_date  115722 non-null  str    
 11  order_estimated_delivery_date  119143 non-null  str    
 12  payment_sequential             119140 non

In [10]:
# checking the null values in the master dataset
df.isnull().sum().sort_values(ascending=False)

review_comment_title       87658
review_comment_message     58274
review_id                      0
review_score                   0
order_id                       0
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [11]:
# filling the null values in the review_comment_title and review_comment_message columns with "No Review"
df["review_comment_title"] = df["review_comment_title"].fillna("No Review")
df["review_comment_message"] = df["review_comment_message"].fillna("No Review")

In [12]:
# checking the null values in the master dataset
customer_df[
    [
        "payment_value",
        "price",
        "freight_value",
        "review_score"
    ]
].isnull().sum()

payment_value      3
price            833
freight_value    833
review_score     997
dtype: int64

In [13]:
# checking the percentage of null values in the master dataset
(customer_df[
    ["payment_value","price","freight_value","review_score"]
]
.isnull()
.mean()*100).round(2)

payment_value    0.00
price            0.70
freight_value    0.70
review_score     0.84
dtype: float64

In [14]:
print(customer_df.shape)
customer_df["customer_unique_id"].nunique()

(119143, 28)


96096

In [15]:
# converting the date columns to datetime format
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_customer_date"
]

for col in date_cols:
    customer_df[col] = pd.to_datetime(customer_df[col])

print(customer_df[date_cols].dtypes)

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_customer_date    datetime64[us]
dtype: object


In [16]:
# creating customer features by aggregating the master dataset on customer_unique_id
customer_features = customer_df.groupby(
    "customer_unique_id"
).agg(
    total_orders=("order_id", "nunique"),
    total_revenue=("payment_value", "sum"),
    avg_order_value=("payment_value", "mean"),
    avg_review_score=("review_score", "mean"),
    total_freight=("freight_value", "sum"),
    avg_installments=("payment_installments", "mean"),
    first_purchase=("order_purchase_timestamp", "min"),
    last_purchase=("order_purchase_timestamp", "max")
).reset_index()

customer_features.head()

,customer_unique_id,total_orders,total_revenue,avg_order_value,avg_review_score,total_freight,avg_installments,first_purchase,last_purchase
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,5.0,12.00,8.0,2018-05-10 10:56:00,2018-05-10 10:56:00
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,4.0,8.29,1.0,2018-05-07 11:11:00,2018-05-07 11:11:00
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,3.0,17.22,8.0,2017-03-10 21:05:00,2017-03-10 21:05:00
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,4.0,17.63,4.0,2017-10-12 20:29:00,2017-10-12 20:29:00
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,5.0,16.89,6.0,2017-11-14 19:45:00,2017-11-14 19:45:00


In [17]:
customer_features.shape

(96096, 9)

In [18]:
# creating new features for customer lifetime and recency
import pandas as pd

reference_date = customer_features["last_purchase"].max()

customer_features["customer_lifetime_days"] = (
    customer_features["last_purchase"]
    - customer_features["first_purchase"]
).dt.days

customer_features["recency_days"] = (
    reference_date
    - customer_features["last_purchase"]
).dt.days

customer_features.head()

,customer_unique_id,total_orders,total_revenue,avg_order_value,avg_review_score,total_freight,avg_installments,first_purchase,last_purchase,customer_lifetime_days,recency_days
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,5.0,12.00,8.0,2018-05-10 10:56:00,2018-05-10 10:56:00,0,160
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,4.0,8.29,1.0,2018-05-07 11:11:00,2018-05-07 11:11:00,0,163
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,3.0,17.22,8.0,2017-03-10 21:05:00,2017-03-10 21:05:00,0,585
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,4.0,17.63,4.0,2017-10-12 20:29:00,2017-10-12 20:29:00,0,369
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,5.0,16.89,6.0,2017-11-14 19:45:00,2017-11-14 19:45:00,0,336


In [19]:
print(customer_features.shape)
customer_features.head()

(96096, 11)


,customer_unique_id,total_orders,total_revenue,avg_order_value,avg_review_score,total_freight,avg_installments,first_purchase,last_purchase,customer_lifetime_days,recency_days
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,5.0,12.00,8.0,2018-05-10 10:56:00,2018-05-10 10:56:00,0,160
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,4.0,8.29,1.0,2018-05-07 11:11:00,2018-05-07 11:11:00,0,163
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,3.0,17.22,8.0,2017-03-10 21:05:00,2017-03-10 21:05:00,0,585
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,4.0,17.63,4.0,2017-10-12 20:29:00,2017-10-12 20:29:00,0,369
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,5.0,16.89,6.0,2017-11-14 19:45:00,2017-11-14 19:45:00,0,336


In [20]:
# creating a (Flag) new feature for repeat customers
customer_features["repeat_customer"] = (
    customer_features["total_orders"] > 1
).astype(int)

customer_features["repeat_customer"].value_counts()

repeat_customer
0    93099
1     2997
Name: count, dtype: int64

In [21]:
# checking class distribution of the repeat_customer feature
customer_features["repeat_customer"].value_counts(normalize=True) * 100

repeat_customer
0    96.881244
1     3.118756
Name: proportion, dtype: float64

In [22]:
# creating a new feature for customer tenure in months
customer_features["customer_tenure_months"] = (
    customer_features["customer_lifetime_days"] / 30
).round(2)
print(customer_features["customer_tenure_months"])

0        0.0
1        0.0
2        0.0
3        0.0
4        0.0
        ... 
96091    0.0
96092    0.0
96093    0.0
96094    0.0
96095    0.0
Name: customer_tenure_months, Length: 96096, dtype: float64


In [23]:
# creating a new feature for revenue per day
customer_features["revenue_per_day"] = (
    customer_features["total_revenue"] /
    (customer_features["customer_lifetime_days"] + 1)
).round(2)

print(customer_features["revenue_per_day"])

0         141.90
1          27.19
2          86.22
3          43.62
4         196.89
          ...   
96091    4134.84
96092      84.58
96093     112.46
96094     133.69
96095      71.56
Name: revenue_per_day, Length: 96096, dtype: float64


In [24]:
# creating a new feature for high value customers based on the 75th percentile of total revenue
revenue_threshold = customer_features["total_revenue"].quantile(0.75)

customer_features["high_value_customer"] = (
    customer_features["total_revenue"] >= revenue_threshold
).astype(int)

customer_features["high_value_customer"].value_counts()

high_value_customer
0    72072
1    24024
Name: count, dtype: int64

In [25]:
import numpy as np
# average freight percentage per customer
customer_features["freight_percentage"] = np.where(
    customer_features["total_revenue"] > 0,
    (
        customer_features["total_freight"]
        / customer_features["total_revenue"]
    ) * 100,
    0
)
print(customer_features["freight_percentage"].round(2))

0         8.46
1        30.49
2        19.97
3        40.42
4         8.58
         ...  
96091    12.03
96092    23.28
96093    20.06
96094    13.98
96095    20.36
Name: freight_percentage, Length: 96096, dtype: float64


In [26]:
# creating a new feature for review category based on average review score
customer_features["review_category"] = pd.cut(
    customer_features["avg_review_score"],
    bins=[0,2,4,5],
    labels=[
        "Poor",
        "Average",
        "Excellent"
    ]
)
print(customer_features["review_category"].value_counts())

review_category
Excellent    54995
Average      26523
Poor         13862
Name: count, dtype: int64


In [27]:
# creating a new feature for recency group based on recency days
customer_features["recency_group"] = pd.cut(
    customer_features["recency_days"],
    bins=[0,30,90,180,365,1000],
    labels=[
        "Very Recent",
        "Recent",
        "Warm",
        "Cold",
        "Inactive"
    ]
)
print(customer_features["recency_group"].value_counts())

recency_group
Cold           39753
Inactive       28362
Warm           18317
Recent          9655
Very Recent        7
Name: count, dtype: int64


In [28]:
customer_features.shape

(96096, 18)

In [29]:
customer_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 96096 entries, 0 to 96095
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   customer_unique_id      96096 non-null  str           
 1   total_orders            96096 non-null  int64         
 2   total_revenue           96096 non-null  float64       
 3   avg_order_value         96095 non-null  float64       
 4   avg_review_score        95380 non-null  float64       
 5   total_freight           96096 non-null  float64       
 6   avg_installments        96095 non-null  float64       
 7   first_purchase          96096 non-null  datetime64[us]
 8   last_purchase           96096 non-null  datetime64[us]
 9   customer_lifetime_days  96096 non-null  int64         
 10  recency_days            96096 non-null  int64         
 11  repeat_customer         96096 non-null  int64         
 12  customer_tenure_months  96096 non-null  float64       
 1

In [30]:
customer_features.head()

,customer_unique_id,total_orders,total_revenue,avg_order_value,avg_review_score,total_freight,avg_installments,first_purchase,last_purchase,customer_lifetime_days,recency_days,repeat_customer,customer_tenure_months,revenue_per_day,high_value_customer,freight_percentage,review_category,recency_group
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,5.0,12.00,8.0,2018-05-10 10:56:00,2018-05-10 10:56:00,0,160,0,0.0,141.90,0,8.456660,Excellent,Warm
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,4.0,8.29,1.0,2018-05-07 11:11:00,2018-05-07 11:11:00,0,163,0,0.0,27.19,0,30.489150,Average,Warm
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,3.0,17.22,8.0,2017-03-10 21:05:00,2017-03-10 21:05:00,0,585,0,0.0,86.22,0,19.972164,Average,Inactive
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,4.0,17.63,4.0,2017-10-12 20:29:00,2017-10-12 20:29:00,0,369,0,0.0,43.62,0,40.417240,Average,Inactive
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,5.0,16.89,6.0,2017-11-14 19:45:00,2017-11-14 19:45:00,0,336,0,0.0,196.89,0,8.578394,Excellent,Cold


In [31]:
# fixing the null values in the customer features dataset
customer_features["avg_review_score"] = (
    customer_features["avg_review_score"]
    .fillna(customer_features["avg_review_score"].median())
)

customer_features["avg_order_value"] = (
    customer_features["avg_order_value"]
    .fillna(0)
)

customer_features["avg_installments"] = (
    customer_features["avg_installments"]
    .fillna(0)
)

customer_features["freight_percentage"] = (
    customer_features["freight_percentage"]
    .fillna(0)
)

In [32]:
# creating RFM scores for customers based on recency, frequency, and monetary value(recency, frequency, monetary)
customer_features["R_score"] = pd.qcut(
    customer_features["recency_days"],
    q=5,
    labels=[5,4,3,2,1]
)

customer_features["F_score"] = pd.qcut(
    customer_features["total_orders"].rank(method="first"),
    q=5,
    labels=[1,2,3,4,5]
)

customer_features["M_score"] = pd.qcut(
    customer_features["total_revenue"],
    q=5,
    labels=[1,2,3,4,5]
)

In [33]:
# creating a new feature for RFM score by concatenating the R, F, and M scores
customer_features["RFM_score"] = (
    customer_features["R_score"].astype(str)
    + customer_features["F_score"].astype(str)
    + customer_features["M_score"].astype(str)
)

print(customer_features["RFM_score"].value_counts())

RFM_score
555    1006
455     974
255     966
355     915
222     894
       ... 
452     672
451     666
325     656
551     654
251     651
Name: count, Length: 125, dtype: int64


In [34]:
# creating a new feature for customer value score by summing the R, F, and M scores
customer_features["customer_value_score"] = (
    customer_features["R_score"].astype(int)
    + customer_features["F_score"].astype(int)
    + customer_features["M_score"].astype(int)
)  

print(customer_features["customer_value_score"].value_counts())

customer_value_score
9     14447
8     13698
10    13482
7     11364
11    11196
6      7856
12     7726
5      4843
13     4694
14     2561
4      2404
15     1006
3       819
Name: count, dtype: int64


In [35]:
customer_features.shape

(96096, 23)

In [36]:
max_order_value=("payment_value","max")
min_order_value=("payment_value","min")
total_reviews=("review_score","count")

In [37]:
agg_dict = {
    "order_id": "nunique",
    "payment_value": ["sum", "mean"]
}

if "review_score" in customer_df.columns:
    agg_dict["review_score"] = "mean"

if "freight_value" in customer_df.columns:
    agg_dict["freight_value"] = "sum"

if "payment_installments" in customer_df.columns:
    agg_dict["payment_installments"] = "mean"

In [38]:
# saving the customer features dataset to a CSV file into database
customer_features.to_sql(
    "customer_features",
    con=engine,
    if_exists="replace",
    index=False
)

96